# UBX Track Plotter
Parses `.ubx` log files and plots the latitude/longitude track.
Uses the same NAV-PVT + NAV-HPPOSLLH merge logic as `v3_ubx_parser.py`.

**Requirements**
```
pip install pyubx2 pandas matplotlib folium
```

In [ ]:
import glob
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors

try:
    import folium
    HAS_FOLIUM = True
except ImportError:
    HAS_FOLIUM = False
    print("folium not installed — interactive map will be skipped. Run: pip install folium")

from pyubx2 import UBXReader

## 1 — Select file
Set `UBX_FILE` to the path of your `.ubx` log, or leave it as `None` to auto-detect the most recent one in this directory.

In [ ]:
UBX_FILE = '/Users/Alexander/Repositories/Ocean_MAE_223/Ocean_Test_Success/dataLog00081.ubx'   # e.g. "../data/dataLog00029.ubx"

if UBX_FILE is None:
    candidates = sorted(glob.glob("../**/*.ubx", recursive=True) + glob.glob("*.ubx"))
    if not candidates:
        sys.exit("No .ubx files found. Set UBX_FILE manually.")
    UBX_FILE = candidates[-1]   # most recent by name

print(f"Parsing: {UBX_FILE}")

## 2 — Parse

In [ ]:
def parse_ubx(path):
    """Returns a DataFrame of position records, one row per epoch.

    pyubx2 (>=1.2.x) auto-scales lat/lon to degrees (float), but leaves
    height / hAcc / vAcc as raw millimetre integers — so we use lat/lon
    as-is and divide the mm fields by 1000 to get metres.
    """
    pvt, hppos = {}, {}

    with open(path, "rb") as f:
        for _, parsed in UBXReader(f):
            if not hasattr(parsed, "identity"):
                continue
            iTOW = getattr(parsed, "iTOW", None)
            if iTOW is None:
                continue

            if parsed.identity == "NAV-PVT":
                pvt[iTOW] = dict(
                    iTOW=iTOW,
                    year=getattr(parsed, "year", 0),
                    month=getattr(parsed, "month", 0),
                    day=getattr(parsed, "day", 0),
                    hour=getattr(parsed, "hour", 0),
                    minute=getattr(parsed, "min", 0),
                    second=getattr(parsed, "sec", 0),
                    fix_type=getattr(parsed, "fixType", 0),
                    carrier_solution=getattr(parsed, "carrSoln", 0),
                    num_sv=getattr(parsed, "numSV", 0),
                    pdop=getattr(parsed, "pDOP", 0) / 100.0,
                    speed_ms=getattr(parsed, "gSpeed", 0) / 1000.0,
                    lat_pvt=float(getattr(parsed, "lat", 0.0)),                # degrees
                    lon_pvt=float(getattr(parsed, "lon", 0.0)),                # degrees
                    h_acc_pvt=getattr(parsed, "hAcc", 0) / 1000.0,             # m
                    height_pvt=getattr(parsed, "height", 0) / 1000.0,          # m above ellipsoid
                    hmsl_pvt=getattr(parsed, "hMSL",   0) / 1000.0,            # m above MSL
                    v_acc_pvt=getattr(parsed, "vAcc",  0) / 1000.0,            # m
                )

            elif parsed.identity == "NAV-HPPOSLLH":
                lat   = float(getattr(parsed, "lat", 0.0))
                lon   = float(getattr(parsed, "lon", 0.0))
                hpLat = float(getattr(parsed, "latHp", getattr(parsed, "hpLat", 0.0)))
                hpLon = float(getattr(parsed, "lonHp", getattr(parsed, "hpLon", 0.0)))
                height = getattr(parsed, "height", 0)                          # mm
                hmsl   = getattr(parsed, "hMSL",   0)                          # mm
                hpHeight = getattr(parsed, "heightHp", getattr(parsed, "hpHeight", 0))  # 0.1 mm
                hpHMSL   = getattr(parsed, "hMSLHp",   getattr(parsed, "hpHMSL",   0))  # 0.1 mm
                hppos[iTOW] = dict(
                    lat_hp=lat + hpLat,
                    lon_hp=lon + hpLon,
                    height_hp=(height + hpHeight * 0.1) / 1000.0,              # m above ellipsoid
                    hmsl_hp=(hmsl   + hpHMSL   * 0.1) / 1000.0,                # m above MSL
                    h_acc_hp=getattr(parsed, "hAcc", 0) / 10000.0,             # 0.1 mm → m
                    v_acc_hp=getattr(parsed, "vAcc", 0) / 10000.0,
                )

    use_hp = len(hppos) > 0
    print(f"NAV-PVT: {len(pvt)}   NAV-HPPOSLLH: {len(hppos)}   "
          f"→ using {'HPPOSLLH' if use_hp else 'PVT'} positions")

    rows = []
    for iTOW in sorted(pvt):
        p = pvt[iTOW].copy()
        hp = hppos.get(iTOW, {})
        if hp:
            p["latitude"]  = hp["lat_hp"]
            p["longitude"] = hp["lon_hp"]
            p["height"]    = hp["height_hp"]
            p["hmsl"]      = hp["hmsl_hp"]
            p["h_acc"]     = hp["h_acc_hp"]
            p["v_acc"]     = hp["v_acc_hp"]
            p["source"]    = "HP"
        else:
            p["latitude"]  = p["lat_pvt"]
            p["longitude"] = p["lon_pvt"]
            p["height"]    = p["height_pvt"]
            p["hmsl"]      = p["hmsl_pvt"]
            p["h_acc"]     = p["h_acc_pvt"]
            p["v_acc"]     = p["v_acc_pvt"]
            p["source"]    = "PVT"
        rows.append(p)

    df = pd.DataFrame(rows)
    df["datetime"] = pd.to_datetime(
        df[["year","month","day","hour","minute","second"]]
        .rename(columns={"minute":"minute","second":"second"}),
        errors="coerce"
    )
    return df


df = parse_ubx(UBX_FILE)
print(f"\n{len(df)} epochs loaded")
print(f"lat range: [{df['latitude'].min():.7f}, {df['latitude'].max():.7f}]  "
      f"Δ = {(df['latitude'].max()-df['latitude'].min())*1e7:.1f}e-7°")
print(f"lon range: [{df['longitude'].min():.7f}, {df['longitude'].max():.7f}]  "
      f"Δ = {(df['longitude'].max()-df['longitude'].min())*1e7:.1f}e-7°")
df[["datetime","latitude","longitude","hmsl","h_acc","v_acc","fix_type","carrier_solution","num_sv"]].head()

## 2b — Isolate the IMU heave window
Restrict the GPS track to the same `t = 800–3200 s` segment analysed in
`PendulumFFT_Simple.ipynb`, assuming the IMU and UBX logs were started together.
Every plot/animation below then covers the same ~40-min ocean-drift segment.

In [ ]:
# ── Isolate the same time window used for the IMU heave analysis ─────────────
# Assumes the IMU and UBX logs were started together (synchronized t = 0).
# PendulumFFT_Simple analysed t = 800–3200 s; restrict the GPS track to match so
# every plot/animation below covers the same ~40-min ocean-drift segment.
WINDOW_START = 800     # s from log start  (= PendulumFFT_Simple T_START)
WINDOW_END   = 3200    # s from log start  (= PendulumFFT_Simple T_END)

df["t_s"] = (df["iTOW"] - df["iTOW"].iloc[0]) / 1000.0   # seconds from first epoch
_n_full = len(df)
df = df[(df["t_s"] >= WINDOW_START) & (df["t_s"] <= WINDOW_END)].reset_index(drop=True)
print(f"Isolated t = {WINDOW_START}-{WINDOW_END} s: kept {len(df)} / {_n_full} epochs "
      f"({df['datetime'].iloc[0]} → {df['datetime'].iloc[-1]})")

## 2c — GPS height vs time (≈1-min slice)
Vertical position from the RTK-fixed epochs over a short ~60 s window — this is the
GPS-measured **wave heave**, directly comparable to the IMU heave spectrum. Adjust
`H_START` / `H_DUR` to slide the window or change its length.

In [ ]:
# ── GPS height vs time over a short (~1 min) slice of the deployment ─────────
H_START = df["t_s"].iloc[0] + 120     # start ~2 min into the isolated window
H_DUR   = 60                          # length of the slice (s)
H_END   = H_START + H_DUR

# clean height: RTK-fixed epochs with good vertical accuracy
sel = (
    (df["t_s"] >= H_START) & (df["t_s"] <= H_END) &
    (df["carrier_solution"] == 2) & (df["v_acc"] <= 0.10)
)
seg = df.loc[sel]
rng = (seg["hmsl"].max() - seg["hmsl"].min()) if len(seg) else float("nan")
print(f"{len(seg)} RTK-fixed epochs in {H_START:.0f}-{H_END:.0f} s  "
      f"(peak-to-trough {rng*100:.0f} cm, std {seg['hmsl'].std()*100:.0f} cm)")

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(seg["t_s"], seg["hmsl"], color="steelblue", lw=1.0, marker=".", ms=3)
ax.set_xlabel("Time since log start (s)")
ax.set_ylabel("Height above MSL (m)")
ax.set_title(f"GPS height vs time  ({H_START:.0f}-{H_END:.0f} s, RTK-fixed)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2d — Slow-drift / tidal-band check
40 min is ~5% of an M2 tidal cycle (12.4 h), so the tidal **frequency** can't be
resolved — but a heavy low-pass that removes the wave band reveals any slow ramp.
A linear fit to the low-passed RTK height gives the mean drift rate (m/hr).

In [ ]:
# ── Very low-pass: look for slow (tidal-band) drift in RTK height ────────────
# 40 min is far too short to RESOLVE a tidal period (~12.4 h), but a heavy
# low-pass that strips the wave band leaves any slow ramp visible. We fit a
# line to the low-passed height to estimate the mean drift rate.
import numpy as np
from scipy.signal import butter, filtfilt

LP_PERIOD = 300.0     # low-pass cutoff period (s) — keep variations slower than this

# clean RTK-fixed height over the (already isolated) in-water window
m = (df["carrier_solution"] == 2) & (df["fix_type"] == 3) & (df["v_acc"] <= 0.05)
d = df.loc[m].reset_index(drop=True)
t = d["t_s"].values
h = d["hmsl"].values
fs_gps = 1.0 / np.median(np.diff(t))

b, a = butter(2, (1.0 / LP_PERIOD) / (fs_gps / 2), "low")
h_lp = filtfilt(b, a, h)

slope, intercept = np.polyfit(t, h_lp, 1)         # m/s, m  (mean drift rate)
drift_cm = slope * (t[-1] - t[0]) * 100
print(f"RTK-clean epochs        : {len(d)} at {fs_gps:.2f} Hz over {(t[-1]-t[0])/60:.1f} min")
print(f"Wave-band std (raw)     : {h.std()*100:.1f} cm")
print(f"Low-pass slow-var range : {(h_lp.max()-h_lp.min())*100:.1f} cm  (T > {LP_PERIOD/60:.0f} min)")
print(f"Median RTK v_acc        : {d['v_acc'].median()*100:.1f} cm")
print(f"Linear drift rate       : {slope*3600:+.3f} m/hr  ({drift_cm:+.1f} cm over the window)")

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(t, h, color="0.8", lw=0.6, label="RTK height (raw)")
ax.plot(t, h_lp, color="C0", lw=1.8, label=f"low-pass (T > {LP_PERIOD/60:.0f} min)")
ax.plot(t, slope * t + intercept, color="C3", lw=1.5, ls="--",
        label=f"linear fit: {slope*3600:+.3f} m/hr")
ax.set_xlabel("Time since log start (s)")
ax.set_ylabel("Height above MSL (m)")
ax.set_title("Slow-drift / tidal-band check (RTK-fixed height, in-water window)")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3 — Fix-quality summary

In [ ]:
FIX_NAMES = {0:"No Fix", 1:"DR", 2:"2D", 3:"3D", 4:"GNSS+DR", 5:"Time"}
RTK_NAMES = {0:"None", 1:"Float", 2:"Fixed"}

print("Fix types:")
print(df["fix_type"].map(FIX_NAMES).value_counts().to_string())
print("\nRTK carrier solution:")
print(df["carrier_solution"].map(RTK_NAMES).value_counts().to_string())
print(f"\nMean h_acc (all):       {df['h_acc'].mean():.3f} m")
rtk = df[df["carrier_solution"] == 2]
if len(rtk):
    print(f"Mean h_acc (RTK Fixed): {rtk['h_acc'].mean()*100:.1f} cm")

## 4 — Matplotlib track plot
Points are colored by RTK carrier solution status.

In [ ]:
RTK_COLORS = {0: "#d62728", 1: "#ff7f0e", 2: "#2ca02c"}   # red / orange / green
RTK_LABELS = {0: "No RTK", 1: "RTK Float", 2: "RTK Fixed"}

fig, ax = plt.subplots(figsize=(8, 6))

# draw track line in light grey first
ax.plot(df["longitude"], df["latitude"], color="#cccccc", linewidth=0.8, zorder=1)

for sol, grp in df.groupby("carrier_solution"):
    ax.scatter(
        grp["longitude"], grp["latitude"],
        c=RTK_COLORS.get(sol, "grey"),
        label=RTK_LABELS.get(sol, str(sol)),
        s=10, zorder=2, alpha=0.8
    )

# mark start / end
ax.plot(df["longitude"].iloc[0],  df["latitude"].iloc[0],  "k^", ms=8, label="Start")
ax.plot(df["longitude"].iloc[-1], df["latitude"].iloc[-1], "ks", ms=8, label="End")

ax.set_xlabel("Longitude (°)")
ax.set_ylabel("Latitude (°)")
ax.set_title(f"GPS Track — {Path(UBX_FILE).name}")
ax.legend(markerscale=1.5, fontsize=9)
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

## 5 — Horizontal accuracy over time

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))

ax.plot(df["iTOW"] / 1000.0, df["h_acc"] * 100, linewidth=0.8, color="steelblue")
ax.set_xlabel("GPS time of week (s)")
ax.set_ylabel("Horizontal accuracy (cm)")
ax.set_title("Horizontal accuracy over time")
ax.set_yscale("log")
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.show()

## 6 — Interactive map (folium)
Points colored by RTK status. Click any point to see timestamp and accuracy.

In [ ]:
if not HAS_FOLIUM:
    print("Install folium to enable this cell: pip install folium")
else:
    center = [df["latitude"].mean(), df["longitude"].mean()]
    fmap = folium.Map(location=center, zoom_start=16, tiles="OpenStreetMap")

    # track polyline
    coords = list(zip(df["latitude"], df["longitude"]))
    folium.PolyLine(coords, color="grey", weight=1.5, opacity=0.6).add_to(fmap)

    for _, row in df.iterrows():
        color = RTK_COLORS.get(int(row["carrier_solution"]), "grey")
        popup = (
            f"{row['datetime']}<br>"
            f"RTK: {RTK_LABELS.get(int(row['carrier_solution']), '?')}<br>"
            f"h_acc: {row['h_acc']*100:.1f} cm<br>"
            f"SVs: {int(row['num_sv'])}"
        )
        folium.CircleMarker(
            location=[row["latitude"], row["longitude"]],
            radius=3, color=color, fill=True, fill_opacity=0.8,
            popup=folium.Popup(popup, max_width=200)
        ).add_to(fmap)

    fmap

## 7 — 3D track plot
Lat/lon/altitude rendered in a local ENU frame (metres east/north of the track centroid),
with altitude on the vertical axis and points colored by RTK status.

In [ ]:
import numpy as np
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (registers 3d projection)


def plot_track_3d(df, alt_col="hmsl",
                  rtk_only=True, require_3d=True,
                  h_acc_max=0.05, v_acc_max=0.10,
                  equal_vertical=False, pad=0.05,
                  line_color="#cccccc"):
    """Plot a 3D GPS track in a local ENU frame (metres E/N, altitude up).

    East and North share one scale so the ground track keeps its true shape.
    Altitude has an independent Z scale by default.

    Filtering
    ---------
    rtk_only=True      → keep only carrier_solution == 2 (RTK Fixed)
    require_3d=True    → keep only fix_type == 3
    h_acc_max / v_acc_max → drop epochs with reported accuracy worse than these (m)

    Set any filter to None (or rtk_only=False) to disable it.
    """
    mask = pd.Series(True, index=df.index)
    if rtk_only:
        mask &= df["carrier_solution"] == 2
    if require_3d:
        mask &= df["fix_type"] == 3
    if h_acc_max is not None:
        mask &= df["h_acc"] <= h_acc_max
    if v_acc_max is not None and alt_col in ("hmsl", "height"):
        mask &= df["v_acc"] <= v_acc_max

    d = df.loc[mask].reset_index(drop=True)
    print(f"filter: kept {len(d)} / {len(df)} epochs "
          f"(rtk_only={rtk_only}, require_3d={require_3d}, "
          f"h_acc_max={h_acc_max}, v_acc_max={v_acc_max})")
    if len(d) < 2:
        print("not enough points after filtering — relax the thresholds.")
        return None

    lat0 = d["latitude"].mean()
    lon0 = d["longitude"].mean()
    alt0 = d[alt_col].mean()

    # local tangent-plane projection (small-area flat-Earth approx)
    R = 6_378_137.0
    east  = np.radians(d["longitude"] - lon0) * R * np.cos(np.radians(lat0))
    north = np.radians(d["latitude"]  - lat0) * R
    up    = d[alt_col] - alt0

    e_span = east.max()  - east.min()
    n_span = north.max() - north.min()
    u_span = up.max()    - up.min()
    print(f"spans after filter: E={e_span*100:.2f} cm  "
          f"N={n_span*100:.2f} cm  alt={u_span*100:.2f} cm")

    h_range = max(e_span, n_span)
    h_half  = max(h_range / 2.0, 0.005) * (1 + pad)   # floor at 5 mm so a true point isn't invisible
    v_half  = max(u_span / 2.0,  0.005) * (1 + pad)
    mid_e, mid_n, mid_u = east.mean(), north.mean(), up.mean()

    fig = plt.figure(figsize=(9, 7))
    ax = fig.add_subplot(111, projection="3d")

    ax.plot(east, north, up, color=line_color, linewidth=0.8, zorder=1)

    for sol, grp in d.groupby("carrier_solution"):
        idx = grp.index
        ax.scatter(
            east.loc[idx], north.loc[idx], up.loc[idx],
            c=RTK_COLORS.get(sol, "grey"),
            label=RTK_LABELS.get(sol, str(sol)),
            s=14, alpha=0.85, depthshade=False, zorder=2,
        )

    ax.scatter(east.iloc[0],  north.iloc[0],  up.iloc[0],
               c="k", marker="^", s=60, label="Start")
    ax.scatter(east.iloc[-1], north.iloc[-1], up.iloc[-1],
               c="k", marker="s", s=60, label="End")

    ax.set_xlim(mid_e - h_half, mid_e + h_half)
    ax.set_ylim(mid_n - h_half, mid_n + h_half)
    if equal_vertical:
        box_half = max(h_half, v_half)
        ax.set_xlim(mid_e - box_half, mid_e + box_half)
        ax.set_ylim(mid_n - box_half, mid_n + box_half)
        ax.set_zlim(mid_u - box_half, mid_u + box_half)
        ax.set_box_aspect((1, 1, 1))
    else:
        ax.set_zlim(mid_u - v_half, mid_u + v_half)
        ax.set_box_aspect((1, 1, 0.6))   # square horizontal, shorter vertical

    ax.set_xlabel("East (m)")
    ax.set_ylabel("North (m)")
    ax.set_zlabel(f"{'MSL' if alt_col == 'hmsl' else 'Ellipsoid'} Δ (m)")
    ax.set_title(
        f"3D GPS Track — {Path(UBX_FILE).name}\n"
        f"origin: ({lat0:.7f}°, {lon0:.7f}°, {alt0:.3f} m)  ·  "
        f"spans E {e_span*100:.1f} cm / N {n_span*100:.1f} cm / alt {u_span*100:.1f} cm"
    )
    ax.legend(fontsize=9, loc="upper left")

    plt.tight_layout()
    plt.show()
    return ax


plot_track_3d(df, alt_col="hmsl");

## 8 — 3D track plot with equal axes
Same plot but with a true 1:1:1 box so E/N/altitude use the same metres-per-unit.

In [ ]:
plot_track_3d(df, alt_col="hmsl", equal_vertical=True);

## 9 — Rotating 3D view (90° animation)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

# Re-filter using the same defaults as plot_track_3d
mask = (
    (df["carrier_solution"] == 2) &
    (df["fix_type"] == 3) &
    (df["h_acc"] <= 0.05) &
    (df["v_acc"] <= 0.10)
)
d = df.loc[mask].reset_index(drop=True)

lat0 = d["latitude"].mean()
lon0 = d["longitude"].mean()
alt0 = d["hmsl"].mean()

R = 6_378_137.0
east  = np.radians(d["longitude"] - lon0) * R * np.cos(np.radians(lat0))
north = np.radians(d["latitude"]  - lat0) * R
up    = d["hmsl"] - alt0

e_span, n_span, u_span = east.max()-east.min(), north.max()-north.min(), up.max()-up.min()
pad = 0.05
h_half = max(max(e_span, n_span) / 2.0, 0.005) * (1 + pad)
v_half = max(u_span / 2.0, 0.005) * (1 + pad)
mid_e, mid_n, mid_u = east.mean(), north.mean(), up.mean()

fig = plt.figure(figsize=(9, 7))
ax  = fig.add_subplot(111, projection="3d")

ax.plot(east, north, up, color="#cccccc", linewidth=0.8, zorder=1)
for sol, grp in d.groupby("carrier_solution"):
    idx = grp.index
    ax.scatter(east.loc[idx], north.loc[idx], up.loc[idx],
               c=RTK_COLORS.get(sol, "grey"), label=RTK_LABELS.get(sol, str(sol)),
               s=14, alpha=0.85, depthshade=False, zorder=2)
ax.scatter(east.iloc[0],  north.iloc[0],  up.iloc[0],  c="k", marker="^", s=60, label="Start")
ax.scatter(east.iloc[-1], north.iloc[-1], up.iloc[-1], c="k", marker="s", s=60, label="End")

ax.set_xlim(mid_e - h_half, mid_e + h_half)
ax.set_ylim(mid_n - h_half, mid_n + h_half)
ax.set_zlim(mid_u - v_half, mid_u + v_half)
ax.set_box_aspect((1, 1, 0.6))
ax.set_xlabel("East (m)")
ax.set_ylabel("North (m)")
ax.set_zlabel("MSL \u0394 (m)")
ax.set_title("3D GPS Track \u2014 rotating 90\u00b0")
ax.legend(fontsize=9, loc="upper left")

init_azim = ax.azim
init_elev = ax.elev

def _rotate(frame):
    ax.view_init(elev=init_elev, azim=init_azim + frame)
    return []

ani = animation.FuncAnimation(
    fig, _rotate, frames=np.arange(0, 91), interval=50, blit=False
)
plt.close()
HTML(ani.to_jshtml())


## 10 — Track fill-in animation (motion over time)
Points accumulate frame-by-frame in chronological order so you can watch the pendulum sweep out its path.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

# ── filter (same defaults as plot_track_3d) ──────────────────────────────────
mask = (
    (df["carrier_solution"] == 2) &
    (df["fix_type"] == 3) &
    (df["h_acc"] <= 0.05) &
    (df["v_acc"] <= 0.10)
)
d = df.loc[mask].reset_index(drop=True)
print(f"{len(d)} points after filter")

# ── project to local ENU (metres) ────────────────────────────────────────────
R = 6_378_137.0
lat0, lon0, alt0 = d["latitude"].mean(), d["longitude"].mean(), d["hmsl"].mean()
east  = (np.radians(d["longitude"] - lon0) * R * np.cos(np.radians(lat0))).values
north = (np.radians(d["latitude"]  - lat0) * R).values
up    = (d["hmsl"] - alt0).values

pad = 0.05
h_half = max((east.max()-east.min()), (north.max()-north.min())) / 2 * (1+pad)
v_half = max((up.max()-up.min()) / 2, 0.005) * (1+pad)
h_half = max(h_half, 0.005)
mid_e, mid_n, mid_u = east.mean(), north.mean(), up.mean()

# ── colour each point by normalized time ─────────────────────────────────────
t_norm = np.linspace(0, 1, len(east))

# ── build figure ─────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(9, 7))
ax  = fig.add_subplot(111, projection="3d")

ax.set_xlim(mid_e - h_half, mid_e + h_half)
ax.set_ylim(mid_n - h_half, mid_n + h_half)
ax.set_zlim(mid_u - v_half, mid_u + v_half)
ax.set_box_aspect((1, 1, 0.6))
ax.set_xlabel("East (m)")
ax.set_ylabel("North (m)")
ax.set_zlabel("MSL \u0394 (m)")
ax.set_title("GPS Track \u2014 filling in over time (purple\u2192yellow)")

# artists updated each frame
trail,   = ax.plot([], [], [], color="#cccccc", linewidth=0.8, zorder=1)
scat     = ax.scatter([], [], [], c=[], cmap="plasma", vmin=0, vmax=1,
                      s=18, depthshade=False, zorder=2)
time_txt = ax.text2D(0.02, 0.95, "", transform=ax.transAxes, fontsize=9)

# ── rotation: one full 360° sweep over the whole animation ───────────────────
init_azim = ax.azim
init_elev = ax.elev

# ── animation: ~120 frames regardless of track length ────────────────────────
N = len(east)
n_frames = min(N, 120)
indices = np.round(np.linspace(1, N, n_frames)).astype(int)

def _update(frame_idx):
    k = indices[frame_idx]
    trail.set_data(east[:k], north[:k])
    trail.set_3d_properties(up[:k])
    scat._offsets3d = (east[:k], north[:k], up[:k])
    scat.set_array(t_norm[:k])
    pct = k / N * 100
    time_txt.set_text(f"epoch {k}/{N}  ({pct:.0f}%)")
    # rotate azimuth one full revolution over all frames
    ax.view_init(elev=init_elev, azim=init_azim + 360 * frame_idx / n_frames)
    return trail, scat, time_txt

ani = animation.FuncAnimation(
    fig, _update, frames=n_frames, interval=80, blit=False
)
plt.close()
HTML(ani.to_jshtml())


## 11 — Save fill-in animation to video
Writes the rotating fill-in animation from cell 10 to an MP4 (requires ffmpeg) with a GIF fallback.

In [ ]:
import shutil, subprocess

stem = Path(UBX_FILE).stem
out_mp4 = Path(stem + "_track_animation.mp4")
out_gif  = Path(stem + "_track_animation.gif")

if shutil.which("ffmpeg"):
    writer = animation.FFMpegWriter(fps=20, bitrate=1800,
                                    extra_args=["-vcodec", "libx264", "-pix_fmt", "yuv420p"])
    ani.save(str(out_mp4), writer=writer, dpi=150)
    print(f"Saved MP4 → {out_mp4.resolve()}")
else:
    print("ffmpeg not found — saving GIF instead (install ffmpeg for MP4 output)")
    ani.save(str(out_gif), writer="pillow", fps=15)
    print(f"Saved GIF → {out_gif.resolve()}")

## 12 — GIF: fill-in animation from 90° rotated view
Same pendulum fill-in as cell 10, but camera fixed at `init_azim + 90°` so you see the orthogonal side of the track.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

# ── reuse filtered ENU data from cell 10 (east / north / up / t_norm) ────────
# (run cell 10 first if variables are not in scope)

fig2 = plt.figure(figsize=(9, 7))
ax2  = fig2.add_subplot(111, projection="3d")

ax2.set_xlim(mid_e - h_half, mid_e + h_half)
ax2.set_ylim(mid_n - h_half, mid_n + h_half)
ax2.set_zlim(mid_u - v_half, mid_u + v_half)
ax2.set_box_aspect((1, 1, 0.6))
ax2.set_xlabel("East (m)")
ax2.set_ylabel("North (m)")
ax2.set_zlabel("MSL Δ (m)")
ax2.set_title("GPS Track — fill-in, 90° rotated view (purple→yellow)")

trail2,  = ax2.plot([], [], [], color="#cccccc", linewidth=0.8, zorder=1)
scat2    = ax2.scatter([], [], [], c=[], cmap="plasma", vmin=0, vmax=1,
                       s=18, depthshade=False, zorder=2)
txt2     = ax2.text2D(0.02, 0.95, "", transform=ax2.transAxes, fontsize=9)

# lock camera at default azimuth + 90°
fixed_azim = ax2.azim - 90
fixed_elev = ax2.elev
ax2.view_init(elev=fixed_elev, azim=fixed_azim)

def _update2(frame_idx):
    k = indices[frame_idx]
    trail2.set_data(east[:k], north[:k])
    trail2.set_3d_properties(up[:k])
    scat2._offsets3d = (east[:k], north[:k], up[:k])
    scat2.set_array(t_norm[:k])
    txt2.set_text(f"epoch {k}/{N}  ({k/N*100:.0f}%)")
    ax2.view_init(elev=fixed_elev, azim=fixed_azim)  # keep fixed
    return trail2, scat2, txt2

ani2 = animation.FuncAnimation(
    fig2, _update2, frames=n_frames, interval=80, blit=False
)
plt.close()

# ── save as GIF ───────────────────────────────────────────────────────────────
out_gif2 = Path(Path(UBX_FILE).stem + "_track_90deg.gif")
ani2.save(str(out_gif2), writer="pillow", fps=15)
print(f"Saved GIF → {out_gif2.resolve()}")

HTML(ani2.to_jshtml())

## 13 — Real-time zoom windows
Define one or more short `(start, end)` time windows (in **seconds from the start of the
filtered log**) and replay each at **real wall-clock speed** so you can watch a very short
slice of the track unfold in real time. Useful for inspecting the slow ocean-drift portions.

* `ZOOM_WINDOWS` — list of `(start_s, end_s)` slices to animate.
* `REALTIME_SPEED` — `1.0` = real time, `0.5` = half speed, `2.0` = 2× faster.
* `VIEW` — `"2d"` for a top-down East/North view, `"3d"` to include altitude.

The data logs at 5 Hz (200 ms/epoch) over ~5060 s total; this file drifts slowly for
most of its length, with the longest continuous drift run around **t ≈ 126–362 s**, so the
defaults below sit inside that stretch.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML, display

# ── Real-time zoom windows ───────────────────────────────────────────────────
# Each entry is a (start, end) pair in SECONDS relative to the first filtered
# epoch in the log.  Every window is replayed at real wall-clock speed so you
# can watch a very short slice of the track unfold in real time.  Add/remove
# windows freely.
ZOOM_WINDOWS = [
    (130, 160),    # 30 s inside the ocean-drift stretch
    (250, 280),    # another 30 s slice
    (1340, 1600),    # a third slice near the end of the long drift run
]
REALTIME_SPEED = 1.0     # 1.0 = real time, 0.5 = half speed, 2.0 = 2x faster
VIEW = "3d"             # "3d" = animated 3D view (like cells 10/12), "2d" = top-down E/N

# ── filter (same defaults as the other animation cells) ──────────────────────
mask = (
    (df["carrier_solution"] == 2) &
    (df["fix_type"] == 3) &
    (df["h_acc"] <= 0.05) &
    (df["v_acc"] <= 0.10)
)
d_all = df.loc[mask].reset_index(drop=True)

# project the whole filtered track to local ENU so all windows share one origin
R = 6_378_137.0
lat0, lon0, alt0 = d_all["latitude"].mean(), d_all["longitude"].mean(), d_all["hmsl"].mean()
east_all  = (np.radians(d_all["longitude"] - lon0) * R * np.cos(np.radians(lat0))).values
north_all = (np.radians(d_all["latitude"]  - lat0) * R).values
up_all    = (d_all["hmsl"] - alt0).values
t_all     = (d_all["iTOW"].values - d_all["iTOW"].values[0]) / 1000.0   # seconds from start
print(f"{len(d_all)} filtered epochs spanning 0 - {t_all[-1]:.1f} s")


def realtime_window_animation(t0, t1):
    """Build a real-time fill-in animation for the epochs in [t0, t1] seconds."""
    sel = (t_all >= t0) & (t_all <= t1)
    n = int(sel.sum())
    if n < 2:
        print(f"window {t0}-{t1}s: only {n} point(s) after filter — skipping")
        return None

    e, no, u, t = east_all[sel], north_all[sel], up_all[sel], t_all[sel]
    t = t - t[0]                      # seconds from window start
    duration = t[-1]                  # real seconds of data in this window

    # frame interval (ms) so playback matches wall-clock time / REALTIME_SPEED
    interval_ms = float(np.median(np.diff(t))) * 1000.0 / REALTIME_SPEED
    cnorm = (t - t[0]) / max(t[-1] - t[0], 1e-9)   # colour by time within window

    pad = 0.10
    if VIEW == "3d":
        fig = plt.figure(figsize=(8, 6))
        ax = fig.add_subplot(111, projection="3d")
        h_half = max(max(e.max()-e.min(), no.max()-no.min()) / 2 * (1+pad), 0.005)
        v_half = max((u.max()-u.min()) / 2, 0.005) * (1+pad)
        ax.set_xlim(e.mean()-h_half, e.mean()+h_half)
        ax.set_ylim(no.mean()-h_half, no.mean()+h_half)
        ax.set_zlim(u.mean()-v_half, u.mean()+v_half)
        ax.set_box_aspect((1, 1, 0.6))
        ax.set_zlabel("MSL Δ (m)")
        trail, = ax.plot([], [], [], color="#cccccc", lw=0.8, zorder=1)
        scat = ax.scatter([], [], [], c=[], cmap="plasma", vmin=0, vmax=1,
                          s=20, depthshade=False, zorder=2)
        head, = ax.plot([], [], [], "o", color="red", ms=9, zorder=3)
        txt = ax.text2D(0.02, 0.95, "", transform=ax.transAxes, fontsize=9)
        init_azim, init_elev = ax.azim, ax.elev   # gentle rotation over the window
    else:
        fig, ax = plt.subplots(figsize=(7, 6))
        ax.set_aspect("equal")
        m = max(max(e.max()-e.min(), no.max()-no.min()) / 2 * (1+pad), 0.005)
        ax.set_xlim(e.mean()-m, e.mean()+m)
        ax.set_ylim(no.mean()-m, no.mean()+m)
        ax.grid(True, alpha=0.3)
        trail, = ax.plot([], [], color="#cccccc", lw=0.8, zorder=1)
        scat = ax.scatter([], [], c=[], cmap="plasma", vmin=0, vmax=1, s=20, zorder=2)
        head, = ax.plot([], [], "o", color="red", ms=9, zorder=3)
        txt = ax.text(0.02, 0.97, "", transform=ax.transAxes, fontsize=9, va="top")

    ax.set_xlabel("East (m)")
    ax.set_ylabel("North (m)")
    ax.set_title(f"Real-time zoom  {t0:.1f}–{t1:.1f}s  "
                 f"({duration:.1f}s of data, {n} pts, {REALTIME_SPEED}× speed)")

    def _update(frame):
        k = frame + 1
        if VIEW == "3d":
            trail.set_data(e[:k], no[:k]); trail.set_3d_properties(u[:k])
            scat._offsets3d = (e[:k], no[:k], u[:k]); scat.set_array(cnorm[:k])
            head.set_data([e[k-1]], [no[k-1]]); head.set_3d_properties([u[k-1]])
            ax.view_init(elev=init_elev, azim=init_azim + 45 * frame / max(n-1, 1))
        else:
            trail.set_data(e[:k], no[:k])
            scat.set_offsets(np.column_stack([e[:k], no[:k]])); scat.set_array(cnorm[:k])
            head.set_data([e[k-1]], [no[k-1]])
        txt.set_text(f"t = {t[k-1]:6.2f}s   epoch {k}/{n}")
        return trail, scat, head, txt

    ani = animation.FuncAnimation(fig, _update, frames=n,
                                  interval=interval_ms, blit=False)
    plt.close()
    return ani


for (t0, t1) in ZOOM_WINDOWS:
    ani = realtime_window_animation(t0, t1)
    if ani is not None:
        display(HTML(ani.to_jshtml()))